# 🔄 Azure AI Search Index 업데이트 - Vector 필드 추가

이 노트북에서는 기존 Azure AI Search 인덱스에 벡터 검색을 위한 필드를 추가합니다.

## 📋 학습 목표

1. **기존 Index 구조 확인하기** - 현재 인덱스 스키마 조회
2. **Vector Search 필드 추가하기** - 임베딩 벡터를 저장할 필드 정의
3. **Vector Search 구성 추가하기** - HNSW 알고리즘 및 프로필 설정
4. **Index 업데이트하기** - 변경사항 적용

## 🔍 주요 개념

### Vector Search란?
- 텍스트를 고차원 벡터(임베딩)로 변환하여 의미적 유사도 기반 검색
- 키워드 매칭이 아닌 **의미론적 유사성**으로 검색
- 예: "노트북" 검색 시 "랩탑", "컴퓨터" 같은 유사 의미도 검색

### 임베딩 모델
- **text-embedding-3-small**: OpenAI의 경량 임베딩 모델
- 벡터 차원: **1536 차원**
- 빠른 속도와 우수한 성능의 균형

### 벡터 검색 알고리즘: ANN vs KNN

#### KNN (K-Nearest Neighbors) - 정확한 검색
- **Exhaustive Search**: 모든 벡터와 거리를 계산
- **100% 정확**: 가장 유사한 결과 보장
- **느린 속도**: 데이터가 많을수록 시간 오래 걸림
- **사용 예**: 소규모 데이터, 높은 정확도가 필수인 경우

#### ANN (Approximate Nearest Neighbors) - 근사 검색
- **HNSW 알고리즘**: 그래프 기반 빠른 탐색
- **95-99% 정확**: KNN과 거의 유사한 결과
- **빠른 속도**: 대규모 데이터에서도 효율적
- **사용 예**: 대부분의 실무 시나리오, 실시간 검색

### 이 노트북에서 추가할 구성
- **HNSW (ANN)**: 빠른 근사 검색용 프로필
- **Exhaustive KNN**: 정확한 전수 검색용 프로필
- 같은 벡터 필드로 두 알고리즘을 비교 가능

---

## 1️⃣ 라이브러리 임포트

Azure AI Search Index 업데이트에 필요한 라이브러리를 임포트합니다.

In [15]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    ExhaustiveKnnAlgorithmConfiguration,
    VectorSearchProfile
)
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
import os

print("✅ 라이브러리 임포트 완료!")

✅ 라이브러리 임포트 완료!


## 2️⃣ 환경 변수 로드 및 클라이언트 초기화

In [16]:
# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 접속 정보
endpoint = os.getenv("SEARCH_ENDPOINT")
key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

# SearchIndexClient 생성
credential = AzureKeyCredential(key)
client = SearchIndexClient(endpoint=endpoint, credential=credential)

print("✅ SearchIndexClient 생성 완료")
print(f"   Endpoint: {endpoint}")
print(f"   Index Name: {index_name}")

✅ SearchIndexClient 생성 완료
   Endpoint: https://foundryiq-aisearch-260114-changjuahn.search.windows.net
   Index Name: products-index


## 3️⃣ 기존 Index 조회

현재 인덱스의 구조를 확인합니다.

In [17]:
# 기존 인덱스 가져오기
index = client.get_index(index_name)

print(f"📋 현재 Index: {index.name}\n")
print("현재 필드 목록:")
for field in index.fields:
    print(f"  - {field.name} ({field.type})")

print(f"\n현재 Vector Search 구성:")
if index.vector_search:
    print("  ✅ Vector Search 설정 존재")
    if index.vector_search.profiles:
        for profile in index.vector_search.profiles:
            print(f"     - Profile: {profile.name}")
else:
    print("  ❌ Vector Search 설정 없음 (추가 필요)")

📋 현재 Index: products-index

현재 필드 목록:
  - id (Edm.String)
  - title (Edm.String)
  - brand (Edm.String)
  - category (Edm.String)
  - normal_price (Edm.Int32)
  - review (Edm.String)

현재 Vector Search 구성:
  ❌ Vector Search 설정 없음 (추가 필요)


## 4️⃣ Vector Search 필드 확인 및 추가

벡터 필드를 확인하고 필요한 필드를 추가합니다.

### 필드 정의
- **content_vector**: 제품명 및 다른 컬럼을 포함한 임베딩 벡터 (1536 차원)
- **review_vector**: 리뷰 임베딩 벡터 (1536 차원)

### ⚠️ 중요
Azure AI Search는 **기존 필드 삭제 불가**합니다.
- 기존 필드가 있으면 그대로 활용
- 없는 필드만 추가


In [ ]:
# 기존 벡터 필드 확인
print("🔍 벡터 필드 확인 중...\n")

existing_field_names = [field.name for field in index.fields]
vector_fields_needed = {
    "content_vector": "제품명 임베딩",
    "review_vector": "리뷰 임베딩"
}

print("현재 벡터 필드 상태:")
for field_name, description in vector_fields_needed.items():
    if field_name in existing_field_names:
        print(f"  ✅ {field_name}: 이미 존재 ({description})")
    else:
        print(f"  ❌ {field_name}: 없음 ({description})")

print()

# 필요한 벡터 필드 추가
print("📝 필요한 벡터 필드 추가 중...\n")

fields_added = []

# content_vector 확인 및 추가
if "content_vector" not in existing_field_names:
    content_vector_field = SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        retrievable=True,  # 핸즈온 학습용: 검색 결과에서 벡터 값 확인 가능
        vector_search_dimensions=1536,
        vector_search_profile_name="hnsw-profile"
    )
    index.fields.append(content_vector_field)
    fields_added.append("content_vector")
    print("✅ content_vector 필드 추가 (제품명 임베딩, 1536차원, retrievable=True)")
else:
    print("ℹ️  content_vector 필드 이미 존재 (재사용)")

# review_vector 확인 및 추가
if "review_vector" not in existing_field_names:
    review_vector_field = SearchField(
        name="review_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        retrievable=True,  # 핸즈온 학습용: 검색 결과에서 벡터 값 확인 가능
        vector_search_dimensions=1536,
        vector_search_profile_name="hnsw-profile"
    )
    index.fields.append(review_vector_field)
    fields_added.append("review_vector")
    print("✅ review_vector 필드 추가 (리뷰 임베딩, 1536차원, retrievable=True)")
else:
    print("ℹ️  review_vector 필드 이미 존재 (재사용)")

print("\n" + "="*60)
if fields_added:
    print(f"✅ {len(fields_added)}개 필드 추가 준비 완료!")
    print("="*60)
    for field in fields_added:
        print(f"   + {field}")
else:
    print("✅ 모든 필드가 이미 존재합니다!")
    print("="*60)
    print("   - content_vector: 제품명 임베딩 (hnsw-profile)")
    print("   - review_vector: 리뷰 임베딩 (hnsw-profile)")


🔍 벡터 필드 확인 중...

현재 벡터 필드 상태:
  ❌ content_vector: 없음 (제품명 임베딩)
  ❌ review_vector: 없음 (리뷰 임베딩)

📝 필요한 벡터 필드 추가 중...

✅ content_vector 필드 추가 (제품명 임베딩, 1536차원)
✅ review_vector 필드 추가 (리뷰 임베딩, 1536차원)

✅ 2개 필드 추가 준비 완료!
   + content_vector
   + review_vector


## 5️⃣ Vector Search 구성 추가

HNSW와 Exhaustive KNN 두 알고리즘을 설정합니다.

### HNSW (Hierarchical Navigable Small World) 파라미터

| 파라미터 | 기본값 | 범위 | 설명 |
|---------|--------|------|------|
| **m** | 4 | 4-10 | **그래프 연결 수** (Bi-directional link count)<br>- 각 노드가 연결하는 이웃 노드 수<br>- 높을수록: 정확도 ↑, 메모리 ↑, 구축 시간 ↑<br>- 낮을수록: 속도 ↑, 메모리 ↓, 정확도 ↓<br>**권장**: 소규모(4), 대규모(6-8) |
| **efConstruction** | 400 | 100-1000 | **인덱스 구축 시 탐색 범위**<br>- 인덱스를 만들 때 탐색할 후보 개수<br>- 높을수록: 인덱스 품질 ↑, 구축 시간 ↑<br>- 낮을수록: 구축 속도 ↑, 품질 ↓<br>**권장**: 일반(400), 고품질(800) |
| **efSearch** | 500 | 100-1000 | **검색 시 탐색 범위**<br>- 검색할 때 탐색할 후보 개수<br>- 높을수록: 정확도 ↑, 검색 시간 ↑<br>- 낮을수록: 속도 ↑, 정확도 ↓<br>**권장**: 일반(500), 고속(200) |

### Exhaustive KNN (Brute Force)
- 모든 벡터와 거리를 계산하여 가장 가까운 K개 선택
- 파라미터: metric만 설정 (m, ef 등 불필요)
- 100% 정확하지만 데이터 많을수록 느림

### 유사도 메트릭 (Similarity Metric)

벡터 간 거리를 측정하는 방식입니다.

#### 1. **cosine** (코사인 유사도) ⭐ 추천
```python
metric: "cosine"
```
- **계산**: 벡터 방향의 유사도 (각도 기반)
- **범위**: -1 ~ 1 (1에 가까울수록 유사)
- **특징**: 
  - 벡터 크기(magnitude)에 영향받지 않음
  - 텍스트 임베딩에 가장 적합
  - 정규화된 벡터에 효과적
- **사용 예**: 텍스트 검색, 문서 유사도, 추천 시스템

#### 2. **dotProduct** (내적)
```python
metric: "dotProduct"
```
- **계산**: 벡터 내적 (방향 + 크기)
- **범위**: -∞ ~ +∞
- **특징**:
  - 벡터 크기를 고려함
  - 정규화된 벡터에서는 cosine과 동일
  - 계산 속도가 가장 빠름
- **사용 예**: 이미 정규화된 임베딩, 빠른 근사 검색

#### 3. **euclidean** (유클리드 거리)
```python
metric: "euclidean"
```
- **계산**: 직선 거리 (√((x₁-x₂)² + (y₁-y₂)² + ...))
- **범위**: 0 ~ ∞ (0에 가까울수록 유사)
- **특징**:
  - 절대적 위치 차이 측정
  - 벡터 크기에 민감
  - 공간적 거리가 중요한 경우 사용
- **사용 예**: 이미지 특징, 좌표 데이터, 센서 데이터

### 메트릭 선택 가이드

| 메트릭 | 텍스트 검색 | 이미지 검색 | 추천 시스템 | 계산 속도 |
|--------|------------|-----------|-----------|----------|
| **cosine** | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | 중간 |
| **dotProduct** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ | 빠름 |
| **euclidean** | ⭐ | ⭐⭐ | ⭐ | 느림 |

**💡 이 프로젝트에서는 `cosine` 사용**
- 텍스트 임베딩 (OpenAI text-embedding-3-small)
- 벡터가 이미 정규화되어 있음
- 방향(의미)이 중요, 크기는 덜 중요

### 예시 코드

```python
# HNSW - 빠른 검색 (기본 설정)
hnsw_config = HnswAlgorithmConfiguration(
    name="hnsw-config",
    parameters={
        "m": 4,                    # 연결 수: 4 (소규모 데이터)
        "efConstruction": 400,     # 구축 품질: 중간
        "efSearch": 500,           # 검색 품질: 중간
        "metric": "cosine"         # 코사인 유사도
    }
)

# HNSW - 고품질 검색 (정확도 우선)
hnsw_high_quality = HnswAlgorithmConfiguration(
    name="hnsw-high-quality",
    parameters={
        "m": 8,                    # 연결 수: 8 (고품질)
        "efConstruction": 800,     # 구축 품질: 높음
        "efSearch": 800,           # 검색 품질: 높음
        "metric": "cosine"
    }
)

# KNN - 정확한 검색
knn_config = ExhaustiveKnnAlgorithmConfiguration(
    name="knn-config",
    parameters={
        "metric": "cosine"         # 코사인 유사도만 설정
    }
)

# dotProduct 사용 예시 (정규화된 벡터)
hnsw_dot = HnswAlgorithmConfiguration(
    name="hnsw-dot",
    parameters={
        "m": 4,
        "efConstruction": 400,
        "efSearch": 500,
        "metric": "dotProduct"     # 내적 사용
    }
)

# euclidean 사용 예시 (공간 데이터)
knn_euclidean = ExhaustiveKnnAlgorithmConfiguration(
    name="knn-euclidean",
    parameters={
        "metric": "euclidean"      # 유클리드 거리
    }
)
```

---

### 이 프로젝트 설정

**HNSW 구성 (빠른 근사 검색)**

In [19]:
# Vector Search 구성이 없으면 새로 생성
if not index.vector_search:
    print("🔧 새로운 Vector Search 구성 생성 중...")
    
    # 1. HNSW 알고리즘 구성 (빠른 근사 검색)
    hnsw_config = HnswAlgorithmConfiguration(
        name="hnsw-config",
        parameters={
            "m": 4,
            "efConstruction": 400,
            "efSearch": 500,
            "metric": "cosine"
        }
    )
    
    # 2. Exhaustive KNN 알고리즘 구성 (정확한 전수 검색)
    knn_config = ExhaustiveKnnAlgorithmConfiguration(
        name="knn-config",
        parameters={
            "metric": "cosine"
        }
    )
    
    # 3. HNSW 프로필
    hnsw_profile = VectorSearchProfile(
        name="hnsw-profile",
        algorithm_configuration_name="hnsw-config"
    )
    
    # 4. KNN 프로필
    knn_profile = VectorSearchProfile(
        name="knn-profile",
        algorithm_configuration_name="knn-config"
    )
    
    # Vector Search 객체 생성 (두 알고리즘, 두 프로필)
    index.vector_search = VectorSearch(
        algorithms=[hnsw_config, knn_config],
        profiles=[hnsw_profile, knn_profile]
    )
    
    print("✅ Vector Search 구성 추가 완료")
    print("   - HNSW 알고리즘 + hnsw-profile (빠름, 근사)")
    print("   - Exhaustive KNN 알고리즘 + knn-profile (느림, 정확)")
else:
    print("✅ Vector Search 구성 이미 존재")
    
    # 기존 알고리즘 확인
    existing_algo_names = [algo.name for algo in (index.vector_search.algorithms or [])]
    existing_profile_names = [profile.name for profile in (index.vector_search.profiles or [])]
    
    # 알고리즘 추가
    if not index.vector_search.algorithms:
        index.vector_search.algorithms = []
    
    if "hnsw-config" not in existing_algo_names:
        hnsw_config = HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters={
                "m": 4,
                "efConstruction": 400,
                "efSearch": 500,
                "metric": "cosine"
            }
        )
        index.vector_search.algorithms.append(hnsw_config)
        print("   + HNSW 알고리즘 추가")
    
    if "knn-config" not in existing_algo_names:
        knn_config = ExhaustiveKnnAlgorithmConfiguration(
            name="knn-config",
            parameters={
                "metric": "cosine"
            }
        )
        index.vector_search.algorithms.append(knn_config)
        print("   + Exhaustive KNN 알고리즘 추가")
    
    # 프로필 추가
    if not index.vector_search.profiles:
        index.vector_search.profiles = []
    
    if "hnsw-profile" not in existing_profile_names:
        hnsw_profile = VectorSearchProfile(
            name="hnsw-profile",
            algorithm_configuration_name="hnsw-config"
        )
        index.vector_search.profiles.append(hnsw_profile)
        print("   + HNSW 프로필 추가")
    
    if "knn-profile" not in existing_profile_names:
        knn_profile = VectorSearchProfile(
            name="knn-profile",
            algorithm_configuration_name="knn-config"
        )
        index.vector_search.profiles.append(knn_profile)
        print("   + KNN 프로필 추가")

🔧 새로운 Vector Search 구성 생성 중...
✅ Vector Search 구성 추가 완료
   - HNSW 알고리즘 + hnsw-profile (빠름, 근사)
   - Exhaustive KNN 알고리즘 + knn-profile (느림, 정확)


## 6️⃣ Index 업데이트 실행

변경사항을 Azure AI Search 서비스에 적용합니다.

⚠️ **주의사항:**
- 인덱스 업데이트는 기존 데이터에 영향을 주지 않습니다
- 새로운 필드는 빈 값으로 시작됩니다
- 다음 단계에서 벡터 값을 채워야 합니다

In [20]:
try:
    # 인덱스 업데이트
    result = client.create_or_update_index(index)
    
    print("="*80)
    print("✅ Index 업데이트 성공!")
    print("="*80)
    print(f"\n📋 Index 이름: {result.name}")
    print(f"📊 총 필드 개수: {len(result.fields)}")
    print("\n전체 필드 목록:")
    for field in result.fields:
        field_type = str(field.type)
        if "Collection" in field_type:
            field_type = f"Vector({field.vector_search_dimensions}D)" if hasattr(field, 'vector_search_dimensions') else "Collection"
        print(f"  - {field.name:20s} : {field_type}")
    
    print("\n🎯 Vector Search 구성:")
    if result.vector_search:
        if result.vector_search.profiles:
            for profile in result.vector_search.profiles:
                print(f"  ✅ Profile: {profile.name}")
        if result.vector_search.algorithms:
            for algo in result.vector_search.algorithms:
                print(f"  ✅ Algorithm: {algo.name}")
    
    print("\n" + "="*80)
    print("🎉 다음 단계: 02-upload_vectors.ipynb에서 벡터 값을 업로드하세요!")
    print("="*80)
    
except Exception as e:
    print(f"❌ Index 업데이트 실패: {str(e)}")

✅ Index 업데이트 성공!

📋 Index 이름: products-index
📊 총 필드 개수: 8

전체 필드 목록:
  - id                   : Edm.String
  - title                : Edm.String
  - brand                : Edm.String
  - category             : Edm.String
  - normal_price         : Edm.Int32
  - review               : Edm.String
  - content_vector       : Vector(1536D)
  - review_vector        : Vector(1536D)

🎯 Vector Search 구성:
  ✅ Profile: hnsw-profile
  ✅ Profile: knn-profile
  ✅ Algorithm: hnsw-config
  ✅ Algorithm: knn-config

🎉 다음 단계: 02-upload_vectors.ipynb에서 벡터 값을 업로드하세요!


---

## ✅ 완료!

인덱스 벡터 필드가 준비되었습니다.

### 📝 변경 사항 요약

1. ✅ **content_vector** 필드 확인/추가 (1536 차원) - 제품명 임베딩
2. ✅ **review_vector** 필드 확인/추가 (1536 차원) - 리뷰 임베딩
3. ✅ **HNSW 알고리즘** 구성 확인
4. ✅ **Exhaustive KNN 알고리즘** 구성 확인
5. ✅ **Vector Search 프로필** 설정 (hnsw-profile, knn-profile)

### 🚀 다음 단계

**02-upload_vectors.ipynb**에서 다음 작업을 수행합니다:
1. 제품명(title)과 리뷰(review) 텍스트를 임베딩 모델로 벡터화
2. 생성된 벡터를 **content_vector**, **review_vector** 필드에 업로드
3. 벡터 검색 준비 완료

### 💡 참고

- **content_vector** = 제품명 임베딩
- **review_vector** = 리뷰 임베딩
- Azure AI Search는 기존 필드 삭제 불가 (제약사항)
- 임베딩 모델: **text-embedding-3-small** (1536 차원)
- 기본 검색 알고리즘: **HNSW** (빠른 근사 검색)
